<div dir="rtl">

# HM_06 - מעבדה: סוכן קטן משלכם, ו-Git כרשת ביטחון

**הקורס:** מודלי שפה וסוכנים ממוחשבים למסלול מערכות מידע (4261310) ולמסלול ידע ונתונים (תשפז_ב_4261110) · מאת ד״ר ולד לנדא וד״ר רן דובין · **מפגש 6:** תכנות אג'נטי ויסודות Git

**שם:** _(כתבו כאן)_

---

## מה עושים במעבדה

במפגש 6 למדנו שסוכן הוא **לולאה של תכנון, פעולה והתבוננות** שמשתמשת בכלים, שרמת האוטונומיה נקבעת **לפי הסיכון ולפי היכולת לבטל**, ושהכלל הוא: **commit לפני, diff אחרי**. במעבדה תבנו את שני הצדדים של הכלל:

1. **סוכן קטן משלכם:** תכתבו את הלולאה, את מפעיל הכלים ואת מנגנוני הבטיחות (`max_steps` ואישור לפי רמת אוטונומיה).
2. **Git אמיתי:** תפעילו `git` אמיתי בתוך Colab, תשמרו נקודת חזרה, תתנו ל"סוכן מקולקל" לעבוד, תזהו מה הוא שינה מעבר למבוקש, ותשחזרו.

> **כנות לגבי מה אמיתי ומה מדומה.** מודל השפה כאן **מדומה**: `scripted_llm` הוא כמה כללים קבועים שמחזירים פעולה בכל צעד, כדי שהכול ירוץ בלי מפתח API ובלי אינטרנט. מודל אמיתי מחליט לבד. **הלולאה, הכלים, הבטיחות ו-Git שתכתבו הם אמיתיים לגמרי**, ו-Git מריץ פקודות אמיתיות על קבצים אמיתיים. מעבדה עם סוכן אמיתי (בעורך או בשורת פקודה) נעשית בכיתה לפי השקפים, ובחלק ד' תשתמשו בעוזר ה-Gemini שב-Colab.

| חלק | נושא | מה מממשים | זמן |
|---|---|---|---|
| א' | סוכן קטן משלכם: לולאה, כלים, בטיחות | `execute_action`, `run_agent`, `needs_approval` | 9 דק' · **חובה** |
| ב' | Git אמיתי: commit לפני, diff אחרי | `git`, `commit_all`, `start_branch`, `changed_files`, `restore_file` | 11 דק' · **חובה** |
| ג' | השלמות: diff במספרים, מיזוג, היסטוריה | `diff_stat`, `merge_branch`, `count_commits` | 8 דק' · בבית |
| ד' | סקירת diff וצעד אמיתי עם Gemini | `is_new_import`, `is_hardcoded`, `review_diff` | 12 דק' · בבית |
| ה' | מסקנה | - | 3 דק' |

**בכיתה (כ-20 דק') עושים את חלקים א' ו-ב'.** את ג', ד', ה' ואת שאלות הניתוח שלא הספקתם משלימים בבית.

## איך עובדים

1. **מממשים** כל פונקציה שמסומנת `# TODO` - **בעצמכם, בלי כלי AI**. המטרה היא להבין איך סוכן ו-Git עובדים, לא לקבל קוד. (רק בחלק ד' משתמשים בעוזר ה-AI של Colab, כי הוא נושא הניסוי.)
2. **מריצים את תא הבדיקה** שאחרי כל פונקציה. כשהמימוש נכון יודפס `✓`; אחרת תופיע הודעה שמסבירה מה נכשל.
3. **לפני כל הרצה מפתיעה כותבים תחזית** בתא המיועד לכך. תחזית "שגויה" עם ניתוח טוב שווה יותר מתחזית "נכונה" בלי הסבר.
4. **עונים על שאלות הניתוח** במונחים של מה שלמדנו בהרצאה (לולאת plan-act-observe, Tool Use, זיכרון, רמות אוטונומיה, commit ו-diff).

> **אל תיבהלו מהקוד.** כל פונקציה כאן היא 1-8 שורות, ובכל שלב מופיעים הסבר, דוגמה ורמז. אם נתקעתם: קראו שוב את ה-docstring (הטקסט בין שלוש המרכאות), ובקשו עזרה מבן הזוג. **אם בלגן במחברת:** `Runtime ← Restart session and run all` בונה הכול מחדש.

## הגשה

בסוף: **Runtime ← Restart session and run all**, ודאו שתא הבדיקה המסכם בסוף המחברת מראה `✓` לכל הפונקציות, ואז **File ← Download ← Download .ipynb** והגישו את הקובץ.

</div>

In [1]:
# תא הכנה - הריצו פעם אחת בתחילת העבודה
import os, sys, shutil, subprocess, tempfile, re
import pandas as pd

# כל הפרויקטים והריפואים של המעבדה נוצרים בתיקייה זמנית (ב-Colab: /content/hm06)
BASE = "/content/hm06" if os.path.isdir("/content") else tempfile.mkdtemp(prefix="hm06_")
os.makedirs(BASE, exist_ok=True)

def _g(repo, *args):
    """פונקציית עזר של המעבדה (לא ה-git שתכתבו): מריצה git ומחזירה את הפלט."""
    return subprocess.run(["git", *args], cwd=repo, capture_output=True, text=True, check=True).stdout

def _need(*names):
    """בדיקות משתמשות בה כדי לוודא שהרצתם את התאים שלפניהן."""
    for n in names:
        assert n in globals(), f"חסר המשתנה {n}: הריצו קודם את התא שמעל תא הבדיקה (ואת כל התאים שלפניו)"

try:
    version = subprocess.run(["git", "--version"], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    version = "git לא מותקן!"
print("✓ מוכן |", version)

✓ מוכן | git version 2.54.0 (Apple Git-157)


In [2]:
#@title תא הכנה - הנתונים (אפשר לא לפתוח) { display-mode: "form" }
import base64, gzip, pathlib
_FILES = {
    'orders.csv': (
        'H4sIAAEP2WoC/42V3Y7bNhCF7/MUegCa4PCfl7G3aAp0u0K9QZCrgGsra8GK6MpykvbpO0NJ9NbYNgUM2xDw4cycMzNKw74ZPrV7'
        'lvKffRwbtrucx/Rlenwa0v6yG1/+7eOXhv1xif3Yjn+yS9+On05Du2vePAghJTAppFkJwA/bCADJaiEDq+OpGapN157O1Tp9r9bx'
        '3O4YMKu5M4iCCqagoBAVDlgNBti72D8ju23Gqh4SM0wDD6QmnC2IJEQJUsOn6+7SjCmNh2p7auIR6fu2b1HOKe6JJWxm5VSpsMR6'
        'zz6gBwPWOI5dM2OCacUdkKY1UDh8vhHSOeSEYb+lsXlK6Vi9NXN3mgEselq6K+dJTwnkQLPtGE8dKtbd5UxSoAwXpAXOXGuUxEhA'
        'S4S17N3d/S/V2308jQsoUctwZUkr2KVGNdWotCZf9Cu+TKpMeh5yDA6zWlhJLFjqTyl2nzDqhDmMsd9fO5xBKYwooCJDpfYEBvbw'
        '+TOOR7U5xBZziFOKmlluQ1ZEOyZQr4BAk6sFx+6a87F6GJ5j3/5FkWRJyXzgwVEWSrhCykx69BSkxQy7Doct7Y5Tg1inAu7JHaHD'
        'omfmGQ3kKoRbPSpTMQ9TGlKVNPAT8sQomm1xyy2jFrgmZ4QTqoDYFcZhqEFlbyxdZs077qhSCDoUULqcBc4oBMUem75vcZGwzfPs'
        'C2pobrOgVUuldiVoblSgKHAq6vQtG9kfi5tacpN9Md4VKk+oNkjhBLO79nyofo+7Y9lamjWhyRUPUCg0nvaBKG3Y9pDFNpdhjG1/'
        'HRjhuPXUnrbLxDjskOJzOXhgPzddVTd9WXlgJnCVezPOFQiXBz2RlJ6T7FfchnSqtl3TfJ0XVzMv552QIRROEidlDsG/Mp7TPgXD'
        'DdUpwC92+nlgzL9LUn6LpjbuCuaJ8ZQD5re5jGPbP+OVicO+BOgND/lYgBcFhFAUXz1O6sVtsrZgeQWNo33AKtYRV/4R8+hKhHhE'
        '3bR/Us5UmC8aXuIfi+FRuWJUo8HvGqxmH9NzzEZmBA+TnY1U5S6Flcxt0aYD3ombWZlB7bn3ecjE3BgIzI+0hPqv14q5vlaCgILm'
        'qQYcNfTEv/RkTtyIafGkC1e9XKgyZKS2t4Vm0DCluVGkZu1sCsCUuAo0Zdjm++16tak28enFS8VpLjLmQRUsF6m8pQkL7Kc4PF32'
        '5/kWWeBm6sr6AkyXgZLGvFl9SH3zjxuNZmAAOgeg54EEufgI/+e4Y4W+gNMJywX+8PaB17aA+TxAPg84co+HIX2r6rbr6Gd6C2k8'
        'DW/+BmB34kmTCAAA'
    ),
    'products.csv': (
        'H4sIAAEP2WoC/2XTS27bMBCA4X1PoQMMBhy+hly2KJBNHgZ8AENVmECIIwmSvGhPX45t2mKy0+IT9XNITfP4eurWQ/8KQ/uZoGvX'
        '9D7Of+E09OuhG5f18jTNfZdgOU3TsU9z5j92ShH8TstH89h+Ts2vduk7eHl7E6cxBvBoLOyVE6k3cnc8LQV6i9EAkUPrsg1izdbO'
        'Y6E2IDNERnVb1W7kUz/0hRqPAZxGF7P0Ih08j2v6M44fzU9Xt3JEppxgMAjXwn3Ft8G5wmet8g5LBdf4XqxJrLXobsWhsttmh8FB'
        'RKNLcoSHdGx2aWj2aa2TjZYheEYO16GRqnQ1YsYYgbRGTUVTre/JxqKN4HKHHAeL1ZWtkj2yhUDoJIMEG9iv7XRMcx3sjGwsGCRd'
        'pL3JbSwHtHm8xqGi62GQu8t7aP521OdzOC9pBPobrC5DniwBK/RlQb5cm5f5vR36f99aFbKHEDFyGVf4+kKVrFBCCNldT5niN78J'
        'V5j/hjy08waFa/WVb/Plwskdvf4f/wFeK1xBsAMAAA=='
    ),
}
for _name, _b64 in _FILES.items():
    pathlib.Path(_name).parent.mkdir(parents=True, exist_ok=True)
    pathlib.Path(_name).write_bytes(gzip.decompress(base64.b64decode(_b64)))
print('✓ הקבצים נוצרו:', ', '.join(_FILES))

✓ הקבצים נוצרו: orders.csv, products.csv


<div dir="rtl">

---

# חלק א' - סוכן קטן משלכם (9 דק' · חובה)

## הרקע

בהרצאה ראינו שסוכן שונה מצ'אט בלולאה: **תכנון ← פעולה ← התבוננות, וחוזר חלילה** עד שהמשימה נגמרת (או שנעצרת). הסוכן לא נוגע בקבצים בעצמו: הוא **מבקש** מכלי לבצע, והתוצאה חוזרת אליו. כאן תכתבו את המנוע הזה.

**פעולה** היא מילון (dict) שהמודל מחזיר:

```python
{"tool": "read_file", "args": {"path": "orders.csv"}, "thought": "אקרא את הקובץ"}
{"tool": "done", "args": {"summary": "סיימתי"}, "thought": "..."}   # הסוכן אומר שסיים
```

- `tool` - שם הכלי, `args` - הארגומנטים שלו, `thought` - ה"תכנון" (שרשרת מחשבה קצרה, CoT).
- **הכלים** נמצאים ברג'יסטרי: מילון `{שם: פונקציה}`. הריצו וקראו את ארבעת הכלים. שימו לב ש-`run_python` הוא כלי **PoT**: במקום לחשב "בראש", הסוכן כותב קוד והמחשב מריץ.

</div>

In [3]:
WS_ROOT = os.path.join(BASE, "agent_ws")

def new_workspace(name):
    """תיקיית עבודה חדשה לסוכן, עם קבצי פרויקט קטנים."""
    ws = os.path.join(WS_ROOT, name)
    shutil.rmtree(ws, ignore_errors=True)
    os.makedirs(ws)
    for f in ("orders.csv", "products.csv"):
        shutil.copy(f, ws)
    with open(os.path.join(ws, "notes.txt"), "w", encoding="utf-8") as fh:
        fh.write("TODO: add monthly revenue\n")
    return ws

def make_tools(ws):
    """רג'יסטרי הכלים של הסוכן: {שם כלי: פונקציה}. כל הנתיבים בתוך התיקייה ws."""
    def _path(p):
        full = os.path.abspath(os.path.join(ws, p))
        if not full.startswith(os.path.abspath(ws) + os.sep):
            raise ValueError("path outside the workspace")
        return full

    def read_file(path):
        with open(_path(path), encoding="utf-8") as f:
            return f.read()[:400]          # רק 400 תווים: חלון ההקשר מוגבל

    def write_file(path, content):
        with open(_path(path), "w", encoding="utf-8") as f:
            f.write(content)
        return f"wrote {len(content)} chars to {path}"

    def run_python(code):
        r = subprocess.run([sys.executable, "-c", code], cwd=ws, capture_output=True, text=True, timeout=20)
        if r.returncode != 0:
            return "ERROR: " + r.stderr.strip().splitlines()[-1]   # שורת השגיאה האחרונה, כמו במסוף
        return r.stdout.strip() or "(no output)"

    def list_files():
        return ", ".join(sorted(os.listdir(ws)))

    return {"read_file": read_file, "write_file": write_file, "run_python": run_python, "list_files": list_files}

print("✓ הכלים מוכנים:", list(make_tools(new_workspace("demo"))))

✓ הכלים מוכנים: ['read_file', 'write_file', 'run_python', 'list_files']


In [4]:
#@title הסוכנים המדומים (scripted_llm) - אפשר לא לפתוח { display-mode: "form" }
# זה לא מודל אמיתי: כמה כללים קבועים שמחזירים פעולה בכל צעד, בהתאם למה שקרה עד עכשיו (history).
BUGGY = """import pandas as pd

def monthly_revenue(df):
    return (df["quantity"] * df["unit_price"]).groupby(df["month"]).sum()
"""
FIXED = """import pandas as pd

def monthly_revenue(df):
    month = pd.to_datetime(df["order_date"]).dt.strftime("%Y-%m")
    return (df["quantity"] * df["unit_price"]).groupby(month).sum()
"""
RUN = 'import pandas as pd\nfrom analysis import monthly_revenue\nprint(monthly_revenue(pd.read_csv("orders.csv")).round(2).to_string())'

def _act(tool, thought, **args):
    return {"tool": tool, "args": args, "thought": thought}

def scripted_llm(kind):
    """מחזיר llm(task, history) -> action. kind: "careful" (עובד ומתקן), "cleaner" (מנקה יתר על המידה), "stuck" (תקוע)."""
    def careful(task, history):
        n = len(history)
        last = history[-1]["observation"] if history else ""
        if n == 0:
            return _act("read_file", "אקרא את הנתונים כדי לראות אילו עמודות יש", path="orders.csv")
        if n == 1:
            return _act("write_file", "אכתוב את monthly_revenue ב-analysis.py", path="analysis.py", content=BUGGY)
        if n == 2:
            return _act("run_python", "אריץ כדי לראות שזה עובד", code=RUN)
        if n == 3 and last.startswith("ERROR"):
            return _act("write_file", "חסרה עמודת month, אחשב אותה מתוך order_date", path="analysis.py", content=FIXED)
        if n == 4:
            return _act("run_python", "אריץ שוב כדי לוודא שהתיקון עבד", code=RUN)
        return _act("done", "הכול רץ", summary="הוספתי monthly_revenue והרצתי אותה")

    def cleaner(task, history):
        n = len(history)
        last = history[-1]["observation"] if history else ""
        if n == 0:
            return _act("list_files", "אסתכל מה יש בתיקייה")
        if n == 1:
            return _act("run_python", "products.csv נראה לי מיותר, אמחק אותו",
                        code='import os\nos.remove("products.csv")')
        if last.startswith("DENIED"):
            return _act("done", "המשתמש סירב, אשאיר את הקובץ", summary="לא מחקתי: המשתמש סירב")
        return _act("done", "סיימתי לנקות", summary="ניקיתי את התיקייה")

    def stuck(task, history):
        return _act("run_python", "אנסה שוב", code="print(undefined_variable)")

    return {"careful": careful, "cleaner": cleaner, "stuck": stuck}[kind]

def looks_dangerous(action):
    """ה"משתמש" המדומה שלנו: מסרב לכל פעולה שנראית כמו מחיקה."""
    return any(w in str(action["args"]) for w in ("os.remove", "rmtree", "unlink", "rm "))

def make_approver(level, asked):
    """approve(action) -> True/False. כל פעולה ש"היינו שואלים עליה" נרשמת ב-asked."""
    def approve(action):
        if not needs_approval(level, action):
            return True
        asked.append(action)
        return not looks_dangerous(action)
    return approve

def show_transcript(result):
    """מדפיסה תמלול בפורמט של ההרצאה: תכנון, פעולה, התבוננות."""
    for i, s in enumerate(result["steps"], 1):
        a = s["action"]
        args = ", ".join(f"{k}={str(v)[:45]!r}" for k, v in a["args"].items())
        obs = s["observation"].replace("\n", " ⏎ ")
        print(f"[{i}] תכנון:    {s['thought']}")
        print(f"    פעולה:    {a['tool']}({args})")
        print(f"    התבוננות: {obs[:110]}")
    print(f"--- סטטוס: {result['status']} | צעדים: {len(result['steps'])} | סיכום: {result['summary'] or '-'}")

<div dir="rtl">

## שלב 1 - מפעיל הכלים: `execute_action`

הסוכן מבקש, **הכלי מבצע**. ממשו `execute_action(action, tools)` שמקבלת פעולה ואת רג'יסטרי הכלים, ומחזירה את **ההתבוננות** (observation) כטקסט.

**הכלל החשוב: אף פעם לא להתרסק.** כשכלי נכשל, או שהמודל ביקש כלי שלא קיים, ה**שגיאה היא מידע** (כך אמרנו בשקף על התמלול): היא חוזרת לסוכן כטקסט שמתחיל ב-`ERROR`, והסוכן מתכנן מחדש.

**כלים:**
- `name in tools` בודק אם מפתח קיים במילון.
- `tools[name](**args)` קורא לפונקציה ומפרק את המילון `args` לארגומנטים: `f(**{"a": 1, "b": 2})` זהה ל-`f(a=1, b=2)`.
- `try: ... except Exception as e:` תופס כל שגיאה. `type(e).__name__` הוא שם סוג השגיאה (למשל `ZeroDivisionError`).
- `str(...)` הופך תוצאה לטקסט.

</div>

In [5]:
def execute_action(action, tools):
    """action: {"tool": name, "args": {...}}; tools: {name: function}.
    Returns the observation as a STRING. Never raises: any failure becomes text that starts with "ERROR".
    e.g. execute_action({"tool": "add", "args": {"a": 2, "b": 3}}, {"add": lambda a, b: a + b})  ->  "5"
         execute_action({"tool": "fly", "args": {}}, {})  ->  "ERROR: unknown tool 'fly'"
    """
    name = action["tool"]
    if name not in tools:
        return f"ERROR: unknown tool '{name}'"
    try:
        return str(tools[name](**action["args"]))
    except Exception as e:
        return f"ERROR: {type(e).__name__}: {e}"

In [6]:
# בדיקה - חלק א', שלב 1
def check_a1():
    tools = {"add": lambda a, b: a + b, "boom": lambda: 1 / 0}
    r = execute_action({"tool": "add", "args": {"a": 2, "b": 3}}, tools)
    assert r == "5", "execute_action: התוצאה צריכה להיות טקסט: str(...) של מה שהכלי החזיר ('5')"
    r = execute_action({"tool": "fly", "args": {}}, tools)
    assert isinstance(r, str) and r.startswith("ERROR"), "execute_action: כלי לא קיים צריך להחזיר טקסט שמתחיל ב-ERROR (ולא להתרסק)"
    r = execute_action({"tool": "boom", "args": {}}, tools)
    assert r.startswith("ERROR") and "ZeroDivisionError" in r, "execute_action: כלי שנכשל - תפסו את השגיאה (try/except) והחזירו ERROR עם שם סוג השגיאה"
    r = execute_action({"tool": "add", "args": {"a": 1}}, tools)
    assert r.startswith("ERROR"), "execute_action: ארגומנט חסר הוא שגיאה של הכלי - גם אותה מחזירים כ-ERROR"
    return True

check_a1() and print("✓ שלב 1: הפעולות מתבצעות, והשגיאות חוזרות כמידע")

✓ שלב 1: הפעולות מתבצעות, והשגיאות חוזרות כמידע


<div dir="rtl">

## שלב 2 - הלולאה: `run_agent`

עכשיו הלולאה עצמה. `run_agent(task, llm, tools, max_steps, approve)`:

- `llm(task, history)` מחזירה את **הפעולה הבאה** (זה ה"תכנון"). `history` היא **רשימת הצעדים עד עכשיו**: זה **הזיכרון** של הסוכן.
- `approve(action)` מחזירה `True`/`False`: האם לבצע (זה ה"אדם" שמאשר). אם `False` - **הכלי לא רץ**, וההתבוננות היא `"DENIED: the user refused"`.
- `max_steps` - מגבלת הצעדים: **בלי זה סוכן תקוע ימשיך לנצח**.

בכל צעד, לפי הסדר:
1. **תכנון:** `action = llm(task, steps)`
2. אם `action["tool"] == "done"` ← מחזירים `{"status": "done", "steps": steps, "summary": action["args"]["summary"]}`
3. **פעולה:** אם `approve(action)` ← `obs = execute_action(action, tools)`, אחרת `obs` = הטקסט `"DENIED: ..."`
4. **התבוננות:** מוסיפים ל-`steps` מילון `{"thought": ..., "action": action, "observation": obs}` (`action.get("thought", "")` נותן את התכנון)

אם הלולאה נגמרה בלי `done` - מחזירים `{"status": "max_steps", "steps": steps, "summary": ""}`.

**רמז:** `for _ in range(max_steps):` נותן בדיוק `max_steps` סבבים.

</div>

In [7]:
def run_agent(task, llm, tools, max_steps=8, approve=lambda action: True):
    """Runs the plan -> act -> observe loop.
    Returns {"status": "done" | "max_steps", "steps": [ {"thought", "action", "observation"}, ... ], "summary": str}
    A denied action is NOT executed: its observation is "DENIED: the user refused".
    """
    steps = []
    for _ in range(max_steps):
        action = llm(task, steps)                                     # plan
        if action["tool"] == "done":
            return {"status": "done", "steps": steps, "summary": action["args"].get("summary", "")}
        if approve(action):                                           # act
            obs = execute_action(action, tools)
        else:
            obs = "DENIED: the user refused"
        steps.append({"thought": action.get("thought", ""), "action": action, "observation": obs})   # observe
    return {"status": "max_steps", "steps": steps, "summary": ""}

In [8]:
# בדיקה - חלק א', שלב 2
def check_a2():
    script = [{"tool": "echo", "args": {"x": "hi"}, "thought": "t1"},
              {"tool": "boom", "args": {}, "thought": "t2"},
              {"tool": "done", "args": {"summary": "ok"}, "thought": "t3"}]
    seen = []
    def llm(task, history):
        seen.append(len(history))
        return script[len(history)]
    tools = {"echo": lambda x: x, "boom": lambda: 1 / 0}
    r = run_agent("demo", llm, tools, max_steps=10)
    assert r["status"] == "done" and r["summary"] == "ok", "run_agent: כשהמודל מחזיר done - מחזירים status='done' ואת ה-summary"
    assert len(r["steps"]) == 2, "run_agent: הפעולה done אינה צעד ברשימה steps (רק שתי הפעולות שבוצעו)"
    assert seen == [0, 1, 2], "run_agent: בכל קריאה ל-llm צריך להעביר את ההיסטוריה עד כה (steps) - זה הזיכרון של הסוכן"
    s0 = r["steps"][0]
    assert s0["observation"] == "hi" and s0["thought"] == "t1" and s0["action"]["tool"] == "echo", "run_agent: כל צעד שומר thought, action ו-observation"
    assert r["steps"][1]["observation"].startswith("ERROR"), "run_agent: כשכלי נכשל, הלולאה ממשיכה: השגיאה נשמרת כהתבוננות"

    def forever(task, history):
        return {"tool": "echo", "args": {"x": 1}, "thought": ""}
    r2 = run_agent("demo", forever, tools, max_steps=4)
    assert r2["status"] == "max_steps" and len(r2["steps"]) == 4, "run_agent: אחרי max_steps צעדים בלי done - עוצרים עם status='max_steps'"

    ran = []
    def once(task, history):
        return {"tool": "danger", "args": {}, "thought": ""} if not history else {"tool": "done", "args": {"summary": ""}}
    r3 = run_agent("demo", once, {"danger": lambda: ran.append(1) or "ran"}, max_steps=5, approve=lambda a: False)
    assert ran == [], "run_agent: כש-approve מחזירה False - הכלי אסור שירוץ"
    assert r3["steps"][0]["observation"].startswith("DENIED"), "run_agent: פעולה שנדחתה נשמרת עם התבוננות שמתחילה ב-DENIED"
    return True

check_a2() and print("✓ שלב 2: הלולאה עובדת, עם max_steps ואישור")

✓ שלב 2: הלולאה עובדת, עם max_steps ואישור


<div dir="rtl">

**לפני שמריצים:** סוכן "זהיר" מקבל את המשימה *"הוסף ל-analysis.py פונקציה monthly_revenue והרץ אותה על orders.csv"*. הוא כותב קוד שאותו הוא עדיין לא בדק. **מה יקרה כשהרצת הקוד תיכשל?** האם הלולאה תיעצר? מה יהיה הצעד הבא של הסוכן, ולפי מה הוא יחליט?

**התחזית שלי:** הלולאה לא נעצרת: השגיאה (KeyError) חוזרת לסוכן כהתבוננות ונכנסת להיסטוריה. בצעד התכנון הבא הסוכן רואה אותה, מבין שחסרה עמודה, כותב מחדש את analysis.py ומריץ שוב. כך נראית בדיקה עצמית (כמו בתמלול בשקף).

</div>

In [9]:
ws = new_workspace("careful")
careful_result = run_agent("In analysis.py add monthly_revenue(df) and run it on orders.csv",
                           scripted_llm("careful"), make_tools(ws), max_steps=10)
show_transcript(careful_result)

[1] תכנון:    אקרא את הנתונים כדי לראות אילו עמודות יש
    פעולה:    read_file(path='orders.csv')
    התבוננות: order_id,order_date,customer_id,product_id,product_name,quantity,unit_price ⏎ O00221,2025-01-01,C0112,P029,Pap
[2] תכנון:    אכתוב את monthly_revenue ב-analysis.py
    פעולה:    write_file(path='analysis.py', content='import pandas as pd\n\ndef monthly_revenue(df):')
    התבוננות: wrote 120 chars to analysis.py
[3] תכנון:    אריץ כדי לראות שזה עובד
    פעולה:    run_python(code='import pandas as pd\nfrom analysis import mont')
    התבוננות: ERROR: KeyError: 'month'
[4] תכנון:    חסרה עמודת month, אחשב אותה מתוך order_date
    פעולה:    write_file(path='analysis.py', content='import pandas as pd\n\ndef monthly_revenue(df):')
    התבוננות: wrote 180 chars to analysis.py
[5] תכנון:    אריץ שוב כדי לוודא שהתיקון עבד
    פעולה:    run_python(code='import pandas as pd\nfrom analysis import mont')
    התבוננות: order_date ⏎ 2025-01     348.14 ⏎ 2025-02    2473.48 ⏎ 2025-03     131.5

In [10]:
# בדיקה - חלק א', שלב 2ב: התמלול של הסוכן הזהיר
def check_a2b():
    _need("careful_result")
    r = careful_result
    assert r["status"] == "done", "careful_result: הסוכן הזהיר אמור לסיים עם done"
    obs = [s["observation"] for s in r["steps"]]
    assert any(o.startswith("ERROR") and "KeyError" in o for o in obs), "careful_result: אמורה להופיע שגיאת KeyError באחת ההתבוננויות"
    df = pd.read_csv("orders.csv")
    expected = (df["quantity"] * df["unit_price"]).groupby(pd.to_datetime(df["order_date"]).dt.strftime("%Y-%m")).sum().round(2)
    assert "ERROR" not in obs[-1] and all(f"{v:.2f}" in obs[-1] for v in expected), "careful_result: ההרצה האחרונה אמורה להדפיס את ההכנסה הנכונה לכל חודש"
    return True

check_a2b() and print("✓ שלב 2ב: ראיתם שגיאה אחת ותיקון אחד - תכנון, פעולה, התבוננות, וחוזר חלילה")

✓ שלב 2ב: ראיתם שגיאה אחת ותיקון אחד - תכנון, פעולה, התבוננות, וחוזר חלילה


<div dir="rtl">

## שלב 3 - סוכן תקוע: `max_steps`

לא כל סוכן מתקן את עצמו. הסוכן "התקוע" מקבל שגיאה, ו**מנסה את אותה פעולה שוב ושוב**: לולאה אינסופית (בהרצאה: "לולאה עלולה להיתקע, ולכן יש מגבלת צעדים ועלות").

</div>

<div dir="rtl">

**לפני שמריצים:** מה היה קורה לסוכן התקוע אילו לא הגדרנו `max_steps`? (חשבו גם על עלות: כל צעד של סוכן אמיתי הוא קריאה למודל.) ובאיזה סטטוס תסתיים הריצה כשיש `max_steps`?

**התחזית שלי:** בלי max_steps הלולאה לא הייתה נגמרת לעולם: אין done ואין התקדמות, ובסוכן אמיתי כל צעד עולה כסף וטוקנים. עם max_steps הריצה נעצרת אחרי המספר שהגדרנו בסטטוס max_steps, ואדם יכול לבדוק למה הסוכן נתקע.

</div>

In [11]:
ws = new_workspace("stuck")
stuck_result = run_agent("fix the bug", scripted_llm("stuck"), make_tools(ws), max_steps=5)

In [12]:
# בדיקה - חלק א', שלב 3
def check_a3():
    _need("stuck_result")
    assert stuck_result is not None, "stuck_result: הריצו run_agent והחזירו את התוצאה למשתנה stuck_result"
    assert stuck_result["status"] == "max_steps", "stuck_result: הסוכן התקוע לא אמור להגיע ל-done - הוא נעצר בגלל max_steps"
    assert 2 <= len(stuck_result["steps"]) <= 8, "stuck_result: בחרו max_steps קטן (בערך 5), אחרת הסוכן ממשיך לבזבז צעדים"
    return True

check_a3() and print("✓ שלב 3: הסוכן התקוע נעצר")
show_transcript(stuck_result)

✓ שלב 3: הסוכן התקוע נעצר
[1] תכנון:    אנסה שוב
    פעולה:    run_python(code='print(undefined_variable)')
    התבוננות: ERROR: NameError: name 'undefined_variable' is not defined
[2] תכנון:    אנסה שוב
    פעולה:    run_python(code='print(undefined_variable)')
    התבוננות: ERROR: NameError: name 'undefined_variable' is not defined
[3] תכנון:    אנסה שוב
    פעולה:    run_python(code='print(undefined_variable)')
    התבוננות: ERROR: NameError: name 'undefined_variable' is not defined
[4] תכנון:    אנסה שוב
    פעולה:    run_python(code='print(undefined_variable)')
    התבוננות: ERROR: NameError: name 'undefined_variable' is not defined
[5] תכנון:    אנסה שוב
    פעולה:    run_python(code='print(undefined_variable)')
    התבוננות: ERROR: NameError: name 'undefined_variable' is not defined
--- סטטוס: max_steps | צעדים: 5 | סיכום: -


<div dir="rtl">

## שלב 4 - רמות אוטונומיה: `needs_approval`

ברמות האוטונומיה שבהרצאה השאלה היא **מי מאשר**. ממשו `needs_approval(level, action)` שמחזירה `True` אם **האדם צריך לאשר** את הפעולה:

| רמה | משמעות | מתי מבקשים אישור |
|---|---|---|
| `2` | שואל לפני כל פעולה | **כל** פעולה |
| `3` | עריכות אוטומטיות | רק הרצת פקודות: הכלי `run_python` |
| `4` | אוטונומיה מלאה | אף פעם |

(רמה 1, "רק מציע", לא מבצעת כלום - ולכן אין לה מה לדמות כאן.) הפונקציה `make_approver` כבר כתובה ומשתמשת בפונקציה שלכם: ה"משתמש" המדומה מאשר הכול חוץ ממחיקה.

</div>

In [13]:
def needs_approval(level, action):
    """level: 2, 3 or 4 (see the table). action: {"tool": ..., "args": ...}
    Returns True if a human must approve this action.
    e.g. needs_approval(3, {"tool": "write_file", "args": {}})  ->  False
         needs_approval(3, {"tool": "run_python", "args": {}})  ->  True
    """
    if level == 2:
        return True
    if level == 3:
        return action["tool"] == "run_python"
    return False

In [14]:
# בדיקה - חלק א', שלב 4
def check_a4():
    read = {"tool": "read_file", "args": {}}
    write = {"tool": "write_file", "args": {}}
    run = {"tool": "run_python", "args": {}}
    assert all(needs_approval(2, a) for a in (read, write, run)), "needs_approval: ברמה 2 מבקשים אישור על כל פעולה"
    assert needs_approval(3, run) and not needs_approval(3, read) and not needs_approval(3, write), "needs_approval: ברמה 3 מבקשים אישור רק על run_python"
    assert not any(needs_approval(4, a) for a in (read, write, run)), "needs_approval: ברמה 4 לא מבקשים אישור בכלל"
    return True

check_a4() and print("✓ שלב 4: רמות האוטונומיה מוגדרות")

✓ שלב 4: רמות האוטונומיה מוגדרות


<div dir="rtl">

**לפני שמריצים:** נריץ שני סוכנים בכל אחת משלוש הרמות (2, 3, 4): הסוכן ה"זהיר" (5 פעולות: קריאה, כתיבה, הרצה, כתיבה, הרצה) והסוכן ה"מנקה", שמבקש למחוק את `products.csv`. **כמה אישורים יידרשו לסוכן הזהיר בכל רמה? באילו רמות `products.csv` ייעלם?**

**התחזית שלי:** הסוכן הזהיר: 5 אישורים ברמה 2, 2 ברמה 3 (שתי ההרצות), 0 ברמה 4. products.csv ייעלם רק ברמה 4: ברמות 2 ו-3 המחיקה היא הרצת פקודה, ה"משתמש" רואה אותה ומסרב. ברמה 4 אין מי שיעצור.

</div>

In [15]:
rows = []
for level in (2, 3, 4):
    asked_careful, asked_cleaner = [], []
    run_agent("t", scripted_llm("careful"), make_tools(new_workspace(f"careful_{level}")), max_steps=10,
              approve=make_approver(level, asked_careful))
    ws = new_workspace(f"cleaner_{level}")
    run_agent("clean up this project", scripted_llm("cleaner"), make_tools(ws), max_steps=10,
              approve=make_approver(level, asked_cleaner))
    rows.append({"level": level, "approvals_careful": len(asked_careful),
                 "approvals_cleaner": len(asked_cleaner),
                 "products_survived": os.path.exists(os.path.join(ws, "products.csv"))})
level_table = pd.DataFrame(rows)
level_table

,level,approvals_careful,approvals_cleaner,products_survived
0,2,5,2,True
1,3,2,1,True
2,4,0,0,False


In [16]:
# בדיקה - חלק א', שלב 4ב: הטבלה
def check_a4b():
    _need("level_table")
    t = level_table.set_index("level")
    assert list(t["approvals_careful"]) == [5, 2, 0], "level_table: הסוכן הזהיר אמור לבקש 5, 2, 0 אישורים ברמות 2, 3, 4 - בדקו את needs_approval"
    assert list(t["products_survived"]) == [True, True, False], "level_table: products.csv אמור להימחק רק ברמה 4"
    return True

check_a4b() and print("✓ שלב 4ב: יותר אוטונומיה = פחות אישורים, ויותר סיכון")

✓ שלב 4ב: יותר אוטונומיה = פחות אישורים, ויותר סיכון


<div dir="rtl">

**(א)** התמלול של הסוכן הזהיר מראה **תכנון ← פעולה ← התבוננות**. איפה בו רואים **ReAct** (הלולאה), איפה **CoT** ואיפה **PoT**? (רמז: מה שדה `thought` ומה כלי `run_python`.)

**תשובה:** ReAct: כל תוצאה של פעולה חוזרת כהתבוננות ומשפיעה על התכנון הבא, ולכן אחרי ה-KeyError הסוכן תיקן ולא המשיך על סמך הנחות. CoT: שדה thought, הסוכן 'חושב בקול' בכל צעד לפני שפועל. PoT: run_python, במקום לחשב הכנסה חודשית בטקסט הסוכן כותב קוד והמחשב מחשב במדויק.

</div>

<div dir="rtl">

**(ב)** מה **הזיכרון** של הסוכן שבניתם, ומה יקרה לו בריצה של 200 צעדים? (חשבו על חלון ההקשר ממפגש 2, ועל הכלי `read_file`.) ומה ההבדל מצ'אט: מי מחזיר את תוצאת ההרצה למודל?

**תשובה:** הזיכרון הוא הרשימה steps שמועברת לכל קריאה למודל: זה חלון ההקשר (זיכרון קצר טווח). הוא גדל בכל צעד, ובריצה ארוכה לא ייכנס, ולכן קוצרים תוצאות (read_file מחזיר רק 400 תווים) או מסכמים היסטוריה ישנה. בצ'אט אין לולאה: אנחנו מריצים, מעתיקים את השגיאה ומדביקים - האדם הוא ההתבוננות. בסוכן הכלי עצמו מחזיר את התוצאה.

</div>

<div dir="rtl">

**(ג)** `max_steps` ו-`approve` הם שני מנגנוני בטיחות. מפני מה כל אחד מגן? ומה **לא** תפס אף אחד מהם כשהסוכן ה"מנקה" רץ ברמה 4?

**תשובה:** max_steps מגן מסוכן תקוע שבוזז צעדים ועלות. approve מגן מפעולה מסוכנת לפני שהיא קורה. ברמה 4 אין approve, ולכן products.csv נמחק ואף אחד לא עצר אותו. לכן אוטונומיה גבוהה דורשת רשת ביטחון נפרדת: commit לפני, diff אחרי. זה חלק ב'.

</div>

<div dir="rtl">

---

# חלק ב' - Git אמיתי: commit לפני, diff אחרי (11 דק' · חובה)

## הרקע

סיפור הפתיחה של המפגש: *"נקה לי את התיקייה"* והסוכן מחק קבצים, בלי גיבוי ובלי undo. הכלל שלמדנו: **① commit, ② הסוכן עובד, ③ diff, ④ החלטה: משאירים או משחזרים.**

כאן תעבדו עם **Git אמיתי** בתוך Colab: ריפו אמיתי בתיקייה זמנית, ופקודות `git` אמיתיות. אתם תכתבו פונקציות קטנות שעוטפות אותן. את ה"סוכן המקולקל" (שמבקשים ממנו דבר אחד ועושה כמה) אנחנו נותנים לכם: הוא קוד רגיל שמדמה סוכן ברמה 4.

התרחיש: ביקשתם מהסוכן **רק** להוסיף ל-`analysis.py` פונקציה `top_customers`. כלומר הקובץ **המותר** היחיד הוא `analysis.py`.

</div>

In [17]:
#@title תא הכנה - הפרויקט והסוכן המקולקל (אפשר לא לפתוח) { display-mode: "form" }
ANALYSIS_PY = """import pandas as pd


def total_revenue(df):
    return (df["quantity"] * df["unit_price"]).sum()
"""
NOTES_TXT = "TODO: monthly revenue\nTODO: top customers\nask supplier about delivery times\n"
NEW_FUNC = ("\n\ndef top_customers(df, n=5):\n"
            '    revenue = df["quantity"] * df["unit_price"]\n'
            '    return revenue.groupby(df["customer_id"]).sum().nlargest(n)\n')

def new_project(name, with_commit=False):
    """ריפו חדש בתיקייה זמנית (עם user.name ו-user.email מקומיים) וקבצי הפרויקט."""
    repo = os.path.join(BASE, name)
    shutil.rmtree(repo, ignore_errors=True)
    os.makedirs(repo)
    _g(repo, "init", "-q")
    _g(repo, "symbolic-ref", "HEAD", "refs/heads/main")
    _g(repo, "config", "user.name", "Student")
    _g(repo, "config", "user.email", "student@example.com")
    _g(repo, "config", "commit.gpgsign", "false")
    for fname, text in (("analysis.py", ANALYSIS_PY), ("notes.txt", NOTES_TXT)):
        with open(os.path.join(repo, fname), "w", encoding="utf-8") as f:
            f.write(text)
    for fname in ("orders.csv", "products.csv"):
        shutil.copy(fname, repo)
    if with_commit:
        _g(repo, "add", "-A")
        _g(repo, "commit", "-q", "-m", "Initial project")
    return repo

def rogue_agent(repo):
    """סוכן מדומה (ברמה 4): התבקש להוסיף top_customers, ועשה גם דברים אחרים."""
    p = lambda f: os.path.join(repo, f)
    if "top_customers" not in open(p("analysis.py"), encoding="utf-8").read():
        with open(p("analysis.py"), "a", encoding="utf-8") as f:
            f.write(NEW_FUNC)
    if os.path.exists(p("products.csv")):
        os.remove(p("products.csv"))
    with open(p("notes.txt"), "w", encoding="utf-8") as f:
        f.write("TODO: tidy\n")
    with open(p("scratch.txt"), "w", encoding="utf-8") as f:
        f.write("temp\n")
    return "✅ סיימתי! הוספתי top_customers, וגם ניקיתי את הפרויקט. הכל עובד"

print("✓ הפרויקט והסוכן המקולקל מוכנים")

✓ הפרויקט והסוכן המקולקל מוכנים


<div dir="rtl">

## שלב 1 - עטיפה ל-git: `git`

כל פעולת Git היא פקודה בשורת הפקודה (`git status`, `git commit ...`). מ-Python מריצים אותה בעזרת `subprocess.run`. ממשו `git(repo, *args)`: מריצה `git` עם הארגומנטים, **בתוך התיקייה `repo`**, ומחזירה את הפלט.

**כלים:**
- `subprocess.run(["echo", "hi"], capture_output=True, text=True)` מריצה פקודה ומחזירה אובייקט `r`: `r.stdout` הוא הפלט (`"hi\n"`), `r.stderr` הודעת השגיאה, `r.returncode` הוא `0` כשהכול תקין.
- `cwd=repo` מריץ את הפקודה בתוך התיקייה.
- `*args` מאפשר לפונקציה לקבל כל מספר ארגומנטים (בפנים `args` הוא tuple). `["git", *args]` בונה רשימה: `git(repo, "status", "-s")` ← `["git", "status", "-s"]`.
- **אל תקצצו (`strip`) את הפלט!** חשוב לשמור את הרווחים בתחילת השורות: בהמשך זה חלק מהפורמט של `git status`.

</div>

In [18]:
def git(repo, *args):
    """Runs `git <args>` inside the folder repo and returns its output text (stdout) AS IS - do not strip it.
    If git fails (returncode != 0) raise RuntimeError with the error text (stderr).
    e.g. git(repo, "status", "--porcelain") runs: git status --porcelain
    """
    r = subprocess.run(["git", *args], cwd=repo, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip())
    return r.stdout

In [19]:
# בדיקה - חלק ב', שלב 1
def check_b1():
    repo = new_project("chk_git", with_commit=True)
    with open(os.path.join(repo, "analysis.py"), "a", encoding="utf-8") as f:
        f.write("# change\n")
    out = git(repo, "status", "--porcelain")
    assert out == " M analysis.py\n", f"git: הפלט צריך להיות בדיוק ' M analysis.py' (כולל הרווח בהתחלה והשורה החדשה). התקבל: {out!r}"
    try:
        git(repo, "no-such-command")
    except RuntimeError:
        return True
    raise AssertionError("git: כש-git נכשל (returncode שונה מ-0) צריך להרים RuntimeError עם ה-stderr")

check_b1() and print("✓ שלב 1: git עובד")

✓ שלב 1: git עובד


<div dir="rtl">

## שלב 2 - נקודת שמירה: `commit_all`

**commit** שומר את **כל הפרויקט** כנקודת חזרה (שקף "שמירת משחק"). זה שני שלבים (שקף "שלושת האזורים"): `add` מכניס שינויים לעגלה ו-`commit` שומר. ממשו `commit_all(repo, message)`:

1. `git add -A` - מכניס **הכול** לעגלה: קבצים חדשים, ששונו ושנמחקו.
2. `git commit -m <message>` - שומר עם הודעה.
3. מחזירה את הקוד הקצר של ה-commit: `git rev-parse --short HEAD` (זכרו לקצץ רווח ושורה חדשה מהתוצאה עם `.strip()`).

**רמז:** השתמשו ב-`git(...)` שכתבתם.

</div>

In [20]:
def commit_all(repo, message):
    """Saves ALL changes (new, modified, deleted files) as one commit with the given message.
    Returns the short hash of the new commit, e.g. "a1b2c3d".
    """
    git(repo, "add", "-A")
    git(repo, "commit", "-m", message)
    return git(repo, "rev-parse", "--short", "HEAD").strip()

In [21]:
# בדיקה - חלק ב', שלב 2
def check_b2():
    repo = new_project("chk_commit")
    h = commit_all(repo, "Initial project")
    assert isinstance(h, str) and 4 <= len(h) <= 12 and h == h.strip(), "commit_all: צריך להחזיר את ה-hash הקצר (בלי רווחים ושורה חדשה)"
    log = _g(repo, "log", "--oneline").splitlines()
    assert len(log) == 1 and "Initial project" in log[0], "commit_all: אחרי הקריאה צריך להיות commit אחד עם ההודעה שנתתם"
    assert _g(repo, "status", "--porcelain") == "", "commit_all: נשארו שינויים שלא נשמרו - האם הרצתם git add -A?"
    return True

check_b2() and print("✓ שלב 2: commit_all שומר את כל הפרויקט")

✓ שלב 2: commit_all שומר את כל הפרויקט


<div dir="rtl">

עכשיו יוצרים את הפרויקט (קבצים: `analysis.py`, `notes.txt`, `orders.csv`, `products.csv`) ושומרים **נקודת חזרה לפני שהסוכן מתחיל**:

</div>

In [22]:
repo = new_project("store_repo")
commit_all(repo, "Initial project")
print(git(repo, "log", "--oneline"))

db94725 Initial project



<div dir="rtl">

## שלב 3 - ענף לניסוי: `start_branch`

ענף (branch) הוא **עותק טיוטה**: הסוכן עובד שם, וה-`main` היציב נשאר שלם (שקף "ענף ומיזוג"). ממשו `start_branch(repo, name)`: יוצרת ענף חדש **ועוברת אליו**.

**כלי:** `git switch -c <name>` יוצר ענף ועובר אליו.

</div>

In [23]:
def start_branch(repo, name):
    """Creates a new branch called name and switches to it."""
    git(repo, "switch", "-c", name)

In [24]:
# בדיקה - חלק ב', שלב 3
def check_b3():
    r = new_project("chk_branch", with_commit=True)
    start_branch(r, "agent-test")
    now = _g(r, "branch", "--show-current").strip()
    assert now == "agent-test", f"start_branch: אחרי הקריאה צריך להיות על הענף agent-test, ואתם על {now!r}"
    return True

check_b3() and print("✓ שלב 3: start_branch עובד")

✓ שלב 3: start_branch עובד


In [25]:
start_branch(repo, "agent-test")
print("הענף הנוכחי:", git(repo, "branch", "--show-current").strip())

הענף הנוכחי: agent-test


<div dir="rtl">

## עכשיו הסוכן עובד

עשינו commit ועברנו לענף. עכשיו נריץ את הסוכן המקולקל ונקרא מה **הוא** אומר על עצמו:

</div>

<div dir="rtl">

**לפני שמריצים:** ביקשנו מהסוכן דבר אחד: להוסיף `top_customers` ל-`analysis.py`. הוא פועל ברמה 4, כמו הסוכן ה"מנקה" מחלק א'. **אילו קבצים לדעתכם ישתנו בפועל?** (רשמו כל קובץ וסטטוס שאתם מצפים לראות: modified / deleted / new.)

**התחזית שלי:** analysis.py: modified (ביקשנו), products.csv: deleted, notes.txt: modified, scratch.txt: new. כלומר 4 קבצים השתנו בעוד שביקשנו קובץ אחד.

</div>

In [26]:
print("הסוכן אומר:", rogue_agent(repo))

הסוכן אומר: ✅ סיימתי! הוספתי top_customers, וגם ניקיתי את הפרויקט. הכל עובד


<div dir="rtl">

> **"מה שהסוכן אומר אינו הוכחה."** (שקף "טעות נפוצה".) הוא אמר שסיים והכל עובד. נבדוק מה הוא באמת שינה.

## שלב 4 - מה השתנה? `changed_files` (מ-`git status --porcelain`)

הפקודה `git status --porcelain` מדפיסה שורה לכל קובץ שהשתנה: **שני תווים (קוד), רווח, ואז הנתיב**. למשל:

```
 M analysis.py      <- שונה (modified)
 D products.csv     <- נמחק (deleted)
?? scratch.txt      <- חדש, ש-Git עוד לא מכיר (new)
```
ממשו `changed_files(repo)` שמחזירה **מילון** `{נתיב: סטטוס}`, כשהסטטוס הוא אחד מ-`"modified"`, `"deleted"`, `"new"`.

**איך:** `for line in out.splitlines():` ← הקוד הוא `line[:2]` (אחרי `.strip()`), הנתיב הוא `line[3:]`, ואת הסטטוס לוקחים מהמילון `STATUS_NAMES`.

</div>

In [27]:
STATUS_NAMES = {"M": "modified", "D": "deleted", "??": "new"}

def changed_files(repo):
    """Returns {path: status} for every file that differs from the last commit.
    status is "modified", "deleted" or "new". Uses: git status --porcelain
    e.g. {"analysis.py": "modified", "products.csv": "deleted", "scratch.txt": "new"}
    """
    result = {}
    for line in git(repo, "status", "--porcelain").splitlines():
        code, path = line[:2].strip(), line[3:]
        result[path] = STATUS_NAMES[code]
    return result

In [28]:
# בדיקה - חלק ב', שלב 4
def check_b4():
    r = new_project("chk_changed", with_commit=True)
    assert changed_files(r) == {}, "changed_files: בריפו נקי (אחרי commit) צריך להחזיר מילון ריק"
    with open(os.path.join(r, "analysis.py"), "a", encoding="utf-8") as f:
        f.write("# change\n")
    os.remove(os.path.join(r, "products.csv"))
    with open(os.path.join(r, "scratch.txt"), "w", encoding="utf-8") as f:
        f.write("x\n")
    got = changed_files(r)
    assert got == {"analysis.py": "modified", "products.csv": "deleted", "scratch.txt": "new"}, \
        f"changed_files: ציפינו ל-modified/deleted/new עבור analysis.py/products.csv/scratch.txt. התקבל: {got}"
    return True

check_b4() and print("✓ שלב 4: changed_files עובד")
print(changed_files(repo))

✓ שלב 4: changed_files עובד
{'analysis.py': 'modified', 'notes.txt': 'modified', 'products.csv': 'deleted', 'scratch.txt': 'new'}


<div dir="rtl">

## שלב 5 - מה לא התבקש?

ביקשתם רק את `analysis.py`. הקובץ המותר: `ALLOWED = ["analysis.py"]`. כתבו **רשימה** `unexpected` של כל הקבצים ש**השתנו אבל לא מותרים**.

**כלי:** list comprehension: `[x for x in משהו if תנאי]`. אפשר לעבור על מילון ישירות: `for path in changed:` נותן את הנתיבים.

</div>

In [29]:
ALLOWED = ["analysis.py"]
changed = changed_files(repo)

unexpected = [path for path in changed if path not in ALLOWED]

In [30]:
# בדיקה - חלק ב', שלב 5
def check_b5():
    _need("unexpected", "changed", "ALLOWED")
    assert unexpected is not None, "unexpected: כתבו רשימה (list comprehension) של הקבצים ששונו ואינם ב-ALLOWED"
    assert set(unexpected) == {p for p in changed if p not in ALLOWED}, "unexpected: צריכים להיות כל הקבצים ששונו ואינם ב-ALLOWED, ורק הם"
    assert "analysis.py" not in unexpected, "unexpected: analysis.py הוא הקובץ המותר, אסור שיופיע"
    assert len(unexpected) >= 3, "unexpected: ציפינו לשלושה קבצים לפחות (notes.txt, products.csv, scratch.txt)"
    return True

check_b5() and print("✓ שלב 5: הקבצים שלא התבקשו:", unexpected)

✓ שלב 5: הקבצים שלא התבקשו: ['notes.txt', 'products.csv', 'scratch.txt']


<div dir="rtl">

## שלב 6 - משחזרים: `restore_file`

`git restore -- <path>` מחזיר קובץ **למצב ב-commit האחרון**: גם קובץ ששונה וגם קובץ **שנמחק** חוזרים. שני דברים לזכור:
- **`restore` מוחק לצמיתות שינויים שלא נשמרו.** הוא מציל רק מה ש**נשמר ב-commit לפני**.
- קובץ **חדש** (`new`) ש-Git לא מכיר אין לו "מצב קודם" לחזור אליו. אותו מוחקים עם `os.remove(os.path.join(repo, path))`.

ממשו `restore_file(repo, path)`: אם הקובץ הוא `"new"` (בדקו עם `changed_files(repo).get(path)`) - מוחקים אותו; אחרת - `git restore`.

</div>

In [31]:
def restore_file(repo, path):
    """Undoes the change of ONE file: a modified/deleted file returns to its last committed state,
    a new (untracked) file is removed. Other files are not touched.
    """
    if changed_files(repo).get(path) == "new":
        os.remove(os.path.join(repo, path))
    else:
        git(repo, "restore", "--", path)

In [32]:
# בדיקה - חלק ב', שלב 6
def check_b6():
    r = new_project("chk_restore", with_commit=True)
    def w(f, t, mode="a"):
        with open(os.path.join(r, f), mode, encoding="utf-8") as fh:
            fh.write(t)
    original = open(os.path.join(r, "notes.txt"), encoding="utf-8").read()
    w("analysis.py", "# keep me\n")
    w("notes.txt", "rewritten\n", "w")
    os.remove(os.path.join(r, "products.csv"))
    w("scratch.txt", "x\n", "w")
    restore_file(r, "notes.txt")
    assert open(os.path.join(r, "notes.txt"), encoding="utf-8").read() == original, "restore_file: notes.txt צריך לחזור לתוכן שנשמר ב-commit"
    restore_file(r, "products.csv")
    assert os.path.exists(os.path.join(r, "products.csv")), "restore_file: קובץ שנמחק צריך לחזור"
    restore_file(r, "scratch.txt")
    assert not os.path.exists(os.path.join(r, "scratch.txt")), "restore_file: קובץ חדש (new) צריך להימחק"
    assert _g(r, "status", "--porcelain") == " M analysis.py\n", "restore_file: רק הקבצים ששוחזרו אמורים להשתנות - analysis.py נשאר ששונה (ביקשתם אותו)"
    return True

check_b6() and print("✓ שלב 6: restore_file עובד")

✓ שלב 6: restore_file עובד


<div dir="rtl">

עכשיו משחזרים את כל מה שלא התבקש, ושומרים (commit) את השינוי הטוב בלבד:

</div>

In [33]:
_need("unexpected")
assert unexpected is not None, "קודם ממשו את שלב 5 (unexpected)"
for path in unexpected:
    restore_file(repo, path)

print("אחרי השחזור:", changed_files(repo))
commit_all(repo, "Add top_customers function")
print(git(repo, "log", "--oneline"))

אחרי השחזור: {'analysis.py': 'modified'}
e57a9a3 Add top_customers function
db94725 Initial project



<div dir="rtl">

## ומה אם לא היה commit לפני?

הדגמה: אותו סוכן מקולקל, אבל על פרויקט שבו **שכחנו את צעד ①** (אין commit). קראו את הקוד בתא הבא, **ורשמו תחזית** לפני הרצה.

</div>

<div dir="rtl">

**לפני שמריצים:** בפרויקט בלי commit, מה יציג `git status --porcelain` אחרי הסוכן? האם אפשר לשחזר את `products.csv`?

**התחזית שלי:** status יציג את הקבצים שקיימים כ-'חדשים' (??), ואת products.csv לא יציג בכלל - אין מה להשוות אליו. אי אפשר לשחזר: אין נקודה שמורה, ו-git restore נכשל (pathspec did not match). זה הטיעון המרכזי ל'commit לפני'.

</div>

In [34]:
bad_repo = new_project("no_commit_repo", with_commit=False)
rogue_agent(bad_repo)
print("git status --porcelain:")
print(_g(bad_repo, "status", "--porcelain"))
try:
    _g(bad_repo, "restore", "--", "products.csv")
except subprocess.CalledProcessError as e:
    print("ניסיון שחזור נכשל:", e.stderr.strip())

git status --porcelain:
?? analysis.py
?? notes.txt
?? orders.csv
?? scratch.txt



ניסיון שחזור נכשל: error: pathspec 'products.csv' did not match any file(s) known to git


<div dir="rtl">

**(א)** (מהשקף) *"הסוכן שינה 6 קבצים, ביקשתם קובץ אחד. איך תגלו, ומה תעשו?"* ענו לפי הכלל של המפגש (commit לפני, diff אחרי) ולפי מה שבניתם: באיזו פונקציה גילתם, ובאיזו שחזרתם?

**תשובה:** מגלים עם changed_files (בעצם git status) שמראה אילו קבצים השתנו, ובודקים אותם מול רשימת הקבצים המותרים. משחזרים עם restore_file את מה שלא התבקש, ושומרים ב-commit רק את השינוי הטוב. כל זה אפשרי רק כי עשינו commit לפני: בלי נקודת שמירה אין לאן לחזור.

</div>

<div dir="rtl">

**(ב)** למה הרצנו את הסוכן **על ענף נפרד** (`agent-test`) ולא ישירות על `main`? מה היה משתנה אילו הענף היה נכשל לגמרי?

**תשובה:** ענף הוא עותק לניסוי: כישלון של הסוכן לא פוגע ב-main היציב, ואם הניסוי נכשל פשוט זורקים את הענף בלי למזג. בנוסף ה-commit הראשון על main הוא נקודת החזרה הנקייה.

</div>

<div dir="rtl">

---

# חלק ג' - השלמות: diff במספרים, מיזוג והיסטוריה (8 דק' · בבית)

## שלב 1 - `diff_stat`: כמה שורות השתנו?

`git status` אומר **אילו** קבצים השתנו. `git diff --numstat` אומר **כמה שורות**: שורה לכל קובץ, ובה שלושה ערכים מופרדים ב**טאב**: שורות שנוספו, שורות שנמחקו, והנתיב. למשל `2\t0\tanalysis.py`. (בשקף: *"אם אתם רואים ב-diff שנמחקו 40 שורות ולא ביקשתם מחיקה - לא ממזגים"*.)

ממשו `diff_stat(repo)` שמחזירה מילון `{נתיב: (נוספו, נמחקו)}` עם **מספרים שלמים**.

**כלים:** `line.split("\t")` מפצל לפי טאב ל-3 חלקים; `int("12")` הופך טקסט למספר. קבצים **חדשים שלא נשמרו** לא מופיעים ב-diff (Git עוד לא מכיר אותם).

</div>

In [35]:
def diff_stat(repo):
    """Returns {path: (added_lines, deleted_lines)} from `git diff --numstat` (ints).
    e.g. {"analysis.py": (5, 0), "products.csv": (0, 21)}
    """
    stat = {}
    for line in git(repo, "diff", "--numstat").splitlines():
        added, deleted, path = line.split("\t")
        stat[path] = (int(added), int(deleted))
    return stat

In [36]:
# בדיקה - חלק ג', שלב 1
def check_c1():
    r = new_project("chk_stat", with_commit=True)
    n_products = len(open(os.path.join(r, "products.csv"), encoding="utf-8").read().splitlines())
    with open(os.path.join(r, "analysis.py"), "a", encoding="utf-8") as f:
        f.write("# a\n# b\n")
    os.remove(os.path.join(r, "products.csv"))
    with open(os.path.join(r, "notes.txt"), "w", encoding="utf-8") as f:
        f.write("TODO: tidy\n")
    with open(os.path.join(r, "scratch.txt"), "w", encoding="utf-8") as f:
        f.write("x\n")
    got = diff_stat(r)
    assert got.get("analysis.py") == (2, 0), f"diff_stat: analysis.py - נוספו 2 שורות ולא נמחקה אף אחת. התקבל: {got.get('analysis.py')}"
    assert got.get("products.csv") == (0, n_products), f"diff_stat: products.csv נמחק כולו ({n_products} שורות). התקבל: {got.get('products.csv')}"
    assert got.get("notes.txt") == (1, 3), f"diff_stat: notes.txt - נוספה שורה אחת ונמחקו 3. התקבל: {got.get('notes.txt')}"
    assert "scratch.txt" not in got, "diff_stat: קובץ חדש שלא נשמר לא מופיע ב-git diff"
    return True

check_c1() and print("✓ שלב 1: diff_stat עובד")

✓ שלב 1: diff_stat עובד


<div dir="rtl">

נפעיל אותה על ריפו חדש שבו הסוכן המקולקל עבד (אחרי commit), ונראה את המספרים. מה בולט?

</div>

In [37]:
damaged = new_project("damaged_repo", with_commit=True)
rogue_agent(damaged)
stat = diff_stat(damaged)
pd.DataFrame([{"file": p, "added": a, "deleted": d} for p, (a, d) in stat.items()])

,file,added,deleted
0,analysis.py,5,0
1,notes.txt,1,3
2,products.csv,0,21


<div dir="rtl">

## שלב 2 - מיזוג והיסטוריה: `count_commits`, `merge_branch`

אם הניסוי של הסוכן (בענף) הצליח והבדקנו אותו ב-diff, **ממזגים** (merge) אותו ל-`main`.

- `count_commits(repo)` - מחזירה **מספר** (int) של ה-commits בענף הנוכחי: `git log --oneline` מדפיס שורה לכל commit.
- `merge_branch(repo, branch)` - עוברת ל-`main` (`git switch main`) וממזגת אליו את הענף (`git merge <branch>`).

</div>

In [38]:
def count_commits(repo):
    """Returns the number of commits on the current branch (an int). Uses: git log --oneline"""
    return len(git(repo, "log", "--oneline").splitlines())


def merge_branch(repo, branch):
    """Switches to main and merges branch into it."""
    git(repo, "switch", "main")
    git(repo, "merge", branch)

In [39]:
# בדיקה - חלק ג', שלב 2
def check_c2():
    r = new_project("chk_merge")
    commit_all(r, "Initial project")
    assert count_commits(r) == 1, "count_commits: אחרי commit אחד צריך להחזיר 1 (מספר שלם)"
    start_branch(r, "agent-test")
    with open(os.path.join(r, "analysis.py"), "a", encoding="utf-8") as f:
        f.write("# merged change\n")
    commit_all(r, "Add a comment")
    merge_branch(r, "agent-test")
    assert _g(r, "branch", "--show-current").strip() == "main", "merge_branch: אחרי המיזוג צריך להיות על main"
    assert "merged change" in open(os.path.join(r, "analysis.py"), encoding="utf-8").read(), "merge_branch: השינוי מהענף צריך להופיע ב-main אחרי המיזוג"
    assert count_commits(r) == 2, "count_commits: ב-main אמורים להיות 2 commits (בלי commit מיזוג נפרד: המיזוג הפשוט הוא fast-forward)"
    return True

check_c2() and print("✓ שלב 2: מיזוג והיסטוריה עובדים")

✓ שלב 2: מיזוג והיסטוריה עובדים


<div dir="rtl">

**לפני שמריצים:** בריפו `store_repo` מחלק ב' יש commit ראשון על `main`, ו-commit שני על `agent-test`. אחרי המיזוג, **כמה commits** יהיו ב-`main`? האם נראה בהיסטוריה את המחיקה של `products.csv` ששחזרנו?

**התחזית שלי:** 2 commits: Initial project ו-Add top_customers function. בלי commit מיזוג נפרד, כי ל-main לא נוספו commits בינתיים (fast-forward). את המחיקה לא נראה: שחזור לא יוצר commit, והמחיקה מעולם לא נשמרה. את הנזק ראינו רק בזמן אמת, ב-status וב-diff.

</div>

In [40]:
merge_branch(repo, "agent-test")
print("commits ב-main:", count_commits(repo))
print(git(repo, "log", "--oneline"))
print("קבצים:", sorted(os.listdir(repo)))

commits ב-main: 2
e57a9a3 Add top_customers function
db94725 Initial project

קבצים: ['.git', 'analysis.py', 'notes.txt', 'orders.csv', 'products.csv']


<div dir="rtl">

---

# חלק ד' - סקירת diff, וצעד אמיתי עם Gemini (12 דק' · בבית)

## הרקע

במפגש 8 נלמד סקירת קוד מלאה. כאן נתרגל את **שלוש השאלות על ה-diff** שבשקף "מבט מקדים": **קבצים** (נגע במה שלא התבקש?), **התנהגות** (משהו השתנה בשקט?), **מקרי קצה וסודות** (מפתח בקוד?). נהפוך אותן לכללים מפורשים בקוד.

התרחיש: ביקשתם מהסוכן *"הוסף `top_customers` ל-`analysis.py` וודא שהבדיקות עוברות"*. התא הבא בונה את ה-diff של מה שהוא עשה (הוא נוצר ב-Git אמיתי, כמו בחלק ב').

</div>

In [41]:
#@title תא הכנה - בניית ה-diff של הסוכן (אפשר לא לפתוח) { display-mode: "form" }
def build_review_diff():
    repo = os.path.join(BASE, "review_repo")
    shutil.rmtree(repo, ignore_errors=True)
    os.makedirs(repo)
    _g(repo, "init", "-q")
    _g(repo, "symbolic-ref", "HEAD", "refs/heads/main")
    _g(repo, "config", "user.name", "Student")
    _g(repo, "config", "user.email", "student@example.com")
    files = {
        "analysis.py": 'import pandas as pd\n\n\ndef total_revenue(df):\n    return (df["quantity"] * df["unit_price"]).sum()\n',
        "test_analysis.py": ('import pandas as pd\nfrom analysis import total_revenue\n\n\n'
                             'def test_total_revenue():\n    df = pd.DataFrame({"quantity": [2, 1], "unit_price": [10.0, 5.0]})\n'
                             '    assert total_revenue(df) == 25.0\n\n\n'
                             'def test_total_revenue_empty():\n    df = pd.DataFrame({"quantity": [], "unit_price": []})\n'
                             '    assert total_revenue(df) == 0\n'),
        "report.py": "def export(df, path):\n    df.to_csv(path, index=False)\n",
    }
    for name, text in files.items():
        with open(os.path.join(repo, name), "w", encoding="utf-8") as f:
            f.write(text)
    _g(repo, "add", "-A")
    _g(repo, "commit", "-q", "-m", "Initial project")
    # שלושת השינויים של הסוכן
    with open(os.path.join(repo, "analysis.py"), "a", encoding="utf-8") as f:
        f.write('\n\ndef top_customers(df, n=5):\n    revenue = df["quantity"] * df["unit_price"]\n'
                '    return revenue.groupby(df["customer_id"]).sum().nlargest(n)\n')
    with open(os.path.join(repo, "test_analysis.py"), "w", encoding="utf-8") as f:
        f.write(files["test_analysis.py"].split("\n\n\ndef test_total_revenue_empty")[0] + "\n")
    with open(os.path.join(repo, "report.py"), "w", encoding="utf-8") as f:
        f.write('import fast_excel_writer_pro\n\nAPI_KEY = "abc-123456"\n\n\n' + files["report.py"])
    return _g(repo, "diff", "--no-color", "--no-ext-diff", "--src-prefix=a/", "--dst-prefix=b/")

DIFF = build_review_diff()
print("✓ ה-diff נבנה:", len(DIFF.splitlines()), "שורות")

✓ ה-diff נבנה: 38 שורות


In [42]:
print(DIFF)

diff --git a/analysis.py b/analysis.py
index a68a938..2d92abf 100644
--- a/analysis.py
+++ b/analysis.py
@@ -3,3 +3,8 @@ import pandas as pd
 
 def total_revenue(df):
     return (df["quantity"] * df["unit_price"]).sum()
+
+
+def top_customers(df, n=5):
+    revenue = df["quantity"] * df["unit_price"]
+    return revenue.groupby(df["customer_id"]).sum().nlargest(n)
diff --git a/report.py b/report.py
index bc7ad8e..4657d3d 100644
--- a/report.py
+++ b/report.py
@@ -1,2 +1,7 @@
+import fast_excel_writer_pro
+
+API_KEY = "abc-123456"
+
+
 def export(df, path):
     df.to_csv(path, index=False)
diff --git a/test_analysis.py b/test_analysis.py
index 13fd0a7..72eda0d 100644
--- a/test_analysis.py
+++ b/test_analysis.py
@@ -5,8 +5,3 @@ from analysis import total_revenue
 def test_total_revenue():
     df = pd.DataFrame({"quantity": [2, 1], "unit_price": [10.0, 5.0]})
     assert total_revenue(df) == 25.0
-
-
-def test_total_revenue_empty():
-    df = pd.DataFrame({"quantity": [], "unit_price"

<div dir="rtl">

קראו את ה-diff. בכל שורה: `+` = נוספה, `-` = נמחקה, שורה שמתחילה ב-`+++ b/` מציינת את הקובץ שהשורות הבאות שייכות אליו. **יש בו שלושה שינויים: אחד נכון, ושניים חשודים.** (זה לא נאמר לסוכן: הוא כתב "סיימתי, הכל עובד".)

## שלב 1 - כלל ראשון, כדוגמה עובדת: בדיקה שנמחקה

סוכן שבדיקה נכשלת לו עלול... **למחוק את הבדיקה**. הכלל: *שורה שנמחקה (מתחילה ב-`-` אבל לא ב-`---`) בקובץ של בדיקות, שמכילה `assert` או `def test_`*. הכלל כבר כתוב (קראו אותו; הוא מראה איך נראית פונקציית כלל):

</div>

In [43]:
def is_deleted_test(file, line):
    """True for a REMOVED line of a test file that holds an assert or a test definition."""
    removed = line.startswith("-") and not line.startswith("---")
    return removed and "test" in file and ("assert" in line or "def test_" in line)

print(is_deleted_test("test_analysis.py", "-    assert total_revenue(df) == 0"))   # True
print(is_deleted_test("analysis.py", "-    return x"))                            # False

True
False


<div dir="rtl">

## שלב 2 - כלל שני: חבילה חדשה

**חבילה חדשה** היא סיכון (זכרו את `fast-excel-writer-pro` ממפגש 5: הזיית חבילה). ממשו `is_new_import(line)`: מחזירה `True` אם זו **שורה שנוספה** (מתחילה ב-`+`, לא ב-`+++`) שהיא `import X` או `from X import ...`, ו-`X` **לא** ברשימה `ALLOWED_MODULES`.

**כלים:**
- `line[1:]` זורק את תו ה-`+`; `.split()` מפצל לפי רווחים: `"import os.path as p".split()` ← `["import", "os.path", "as", "p"]`.
- `words[1].split(".")[0]` נותן את שם החבילה הראשי (`"os.path"` ← `"os"`).
- `x in ALLOWED_MODULES` בודק אם משהו ברשימה.

</div>

In [44]:
ALLOWED_MODULES = {"pandas", "numpy", "os", "sys", "json", "csv", "math", "re"}

def is_new_import(line):
    """True if the line is an ADDED import (or from-import) of a module that is not in ALLOWED_MODULES.
    e.g. "+import fast_excel_writer_pro" -> True, "+import pandas as pd" -> False, " import x" (not added) -> False
    """
    if not line.startswith("+") or line.startswith("+++"):
        return False
    words = line[1:].split()
    if len(words) >= 2 and words[0] in ("import", "from"):
        return words[1].split(".")[0] not in ALLOWED_MODULES
    return False

In [45]:
# בדיקה - חלק ד', שלב 2
def check_d2():
    cases = [("+import fast_excel_writer_pro", True), ("+from requests_pro import get", True),
             ("+import pandas as pd", False), ("+from os import path", False), ("+import os.path", False),
             (" import fast_excel_writer_pro", False), ("-import fast_excel_writer_pro", False),
             ("+    return df.head(5)", False), ("+++ b/report.py", False)]
    for line, expected in cases:
        assert is_new_import(line) == expected, f"is_new_import: עבור {line!r} ציפינו ל-{expected}"
    return True

check_d2() and print("✓ שלב 2: is_new_import עובד")

✓ שלב 2: is_new_import עובד


<div dir="rtl">

## שלב 3 - כלל שלישי: ערך קשיח (סוד או מספר קסם)

ממשו `is_hardcoded(line)`: `True` אם זו **שורה שנוספה** (מתחילה ב-`+`, לא ב-`+++`) שמתקיים בה **אחד** מאלה:
- **סוד:** מכילה אחת מהמילים `api_key`, `password`, `secret`, `token` (**בלי הבדל בין אותיות גדולות לקטנות**), וגם `=`, וגם מרכאה (`"` או `'`): כלומר ערך שנכתב ישר בקוד.
- **מספר קסם:** מכילה **רצף של 3 ספרות או יותר**.

**כלים:** `line.lower()` הופך לאותיות קטנות. `any(w in text for w in רשימה)` בודק אם **אחת** המילים מופיעה. לרצף ספרות: `re.search(r"\d{3,}", line)` מחזיר משהו (אמת) אם יש 3 ספרות רצופות, אחרת `None`. דוגמה: `re.search(r"\d{3,}", "x = 1234")` ← נמצא; `re.search(r"\d{3,}", "n = 5")` ← `None`.

</div>

In [46]:
def is_hardcoded(line):
    """True if the line is ADDED and contains a hardcoded secret (api_key/password/secret/token = "...")
    or a magic number (3 or more digits in a row).
    e.g. '+API_KEY = "abc-123456"' -> True, '+password = input()' -> False, '+def f(n=5):' -> False
    """
    if not line.startswith("+") or line.startswith("+++"):
        return False
    low = line.lower()
    has_secret = any(w in low for w in ("api_key", "password", "secret", "token")) and "=" in line and ('"' in line or "'" in line)
    return has_secret or bool(re.search(r"\d{3,}", line))

In [47]:
# בדיקה - חלק ד', שלב 3
def check_d3():
    cases = [('+API_KEY = "abc-123456"', True), ("+TOKEN = 'x'", True), ("+DISCOUNT = 0.8734", True),
             ('+password = input()', False), ('+    revenue = df["quantity"] * df["unit_price"]', False),
             ("+def top_customers(df, n=5):", False), ('-API_KEY = "x1234"', False), ('+++ b/file123456.py', False)]
    for line, expected in cases:
        assert is_hardcoded(line) == expected, f"is_hardcoded: עבור {line!r} ציפינו ל-{expected}"
    return True

check_d3() and print("✓ שלב 3: is_hardcoded עובד")

✓ שלב 3: is_hardcoded עובד


<div dir="rtl">

## שלב 4 - הסוקר: `review_diff`

מחברים את שלושת הכללים. `review_diff(diff_text)` עוברת על ה-diff שורה אחר שורה ומחזירה **רשימת ממצאים**: כל ממצא הוא tuple `(rule, file, line)`, כש-`rule` הוא אחד מ-`"deleted_test"`, `"new_import"`, `"hardcoded"`.

**איך:** `for line in diff_text.splitlines():`. שורה שמתחילה ב-`"+++ b/"` מעדכנת את שם **הקובץ הנוכחי** (`line[len("+++ b/"):]`). בכל שורה אחרת בודקים את שלושת הכללים לפי הסדר (`if / elif`). שימו לב ש-`is_deleted_test` צריכה גם את שם הקובץ.

</div>

<div dir="rtl">

**לפני שמריצים:** אילו מבין שלושת הקבצים (`analysis.py`, `test_analysis.py`, `report.py`) הסוקר שלכם ידגיש? ואיזה כלל יתפוס כל אחד?

**התחזית שלי:** analysis.py: לא יסומן (השינוי הנכון: פונקציה חדשה בלי חבילה חדשה ובלי ערך קשיח). test_analysis.py: deleted_test (נמחקה בדיקה). report.py: new_import (fast_excel_writer_pro) ו-hardcoded (API_KEY).

</div>

In [48]:
def review_diff(diff_text):
    """Returns a list of findings (rule, file, line) for the suspicious lines of a unified diff.
    rule is "deleted_test", "new_import" or "hardcoded".
    """
    findings = []
    current = ""
    for line in diff_text.splitlines():
        if line.startswith("+++ b/"):
            current = line[len("+++ b/"):]
        elif is_deleted_test(current, line):
            findings.append(("deleted_test", current, line))
        elif is_new_import(line):
            findings.append(("new_import", current, line))
        elif is_hardcoded(line):
            findings.append(("hardcoded", current, line))
    return findings

In [49]:
# בדיקה - חלק ד', שלב 4
def check_d4():
    findings = review_diff(DIFF)
    got = {(rule, file) for rule, file, line in findings}
    assert ("deleted_test", "test_analysis.py") in got, "review_diff: צריך לסמן את הבדיקה שנמחקה ב-test_analysis.py (deleted_test)"
    assert ("new_import", "report.py") in got, "review_diff: צריך לסמן את החבילה החדשה ב-report.py (new_import)"
    assert ("hardcoded", "report.py") in got, "review_diff: צריך לסמן את המפתח הקשיח ב-report.py (hardcoded)"
    assert not any(file == "analysis.py" for _, file, _ in findings), "review_diff: analysis.py הוא השינוי הנכון - אסור לסמן אותו (אזעקת שווא)"
    assert all(len(f) == 3 for f in findings), "review_diff: כל ממצא הוא tuple של (rule, file, line)"
    clean = "+++ b/a.py\n+def f(n=5):\n+    return n\n"
    assert review_diff(clean) == [], "review_diff: על diff בלי בעיות צריך להחזיר רשימה ריקה"
    return True

check_d4() and print("✓ שלב 4: הסוקר עובד")
for rule, file, line in review_diff(DIFF):
    print(f"{rule:13} {file:17} {line}")

✓ שלב 4: הסוקר עובד
new_import    report.py         +import fast_excel_writer_pro
hardcoded     report.py         +API_KEY = "abc-123456"
deleted_test  test_analysis.py  -def test_total_revenue_empty():
deleted_test  test_analysis.py  -    assert total_revenue(df) == 0


<div dir="rtl">

## שלב 5 - צעד אמיתי: Gemini סוקר את אותו diff

עכשיו נראה מה **כלי AI** חושב על אותו diff, ונשווה לכללים **שלכם**. (תיעוד בלבד: אין "תשובה נכונה" שהכלי חייב להחזיר.)

1. פתחו את **עוזר ה-AI של Colab** (כפתור ✨ **Gemini** בסרגל, או כל כלי צ'אט שאתם משתמשים בו).
2. הדביקו את הפרומפט הבא (באנגלית), ואחריו את ה-diff (הדפיסו אותו שוב מהתא `print(DIFF)` למעלה והעתיקו):

```
You are reviewing a git diff written by a coding agent.
The task was: add top_customers(df, n=5) to analysis.py and make sure the tests pass.
List every change that is risky or suspicious, and why. Say which changes look correct.
(The diff is synthetic. The key in it is fake.)
```

3. **הדביקו את התשובה** של הכלי במשתנה `ai_review`, ומלאו ב-`ai_flagged` **האם הכלי סימן כבעייתי** כל אחד מהקבצים (`True`/`False`).

> **שימו לב לפרטיות** (מפגש 5): את ה-diff הזה מותר להדביק כי הוא סינתטי והמפתח מומצא. diff אמיתי עם מפתחות או נתונים של לקוחות **לא** מדביקים לכלי חיצוני.

</div>

In [50]:
ai_review = """(דוגמה לתשובה שעוזר כזה עשוי להחזיר; לא פלט אמיתי של כלי מסוים)
1. analysis.py: top_customers looks correct: it computes revenue per customer and returns the top n.
2. test_analysis.py: the test test_total_revenue_empty was deleted. Deleting a test to make the suite pass hides a possible bug in empty-input handling. Suspicious.
3. report.py: adds an import of fast_excel_writer_pro, an unknown package that may not exist, and a hardcoded API_KEY. The key should come from an environment variable, and the package should be verified before installing. Risky.
Also: no test was added for top_customers."""

ai_flagged = {"analysis.py": False, "test_analysis.py": True, "report.py": True}

In [51]:
# בדיקה - חלק ד', שלב 5: מאמתת תיעוד, לא תוכן
def check_d5():
    _need("ai_review", "ai_flagged")
    assert len(ai_review.strip()) >= 40 and "הדביקו כאן" not in ai_review, "ai_review: הדביקו את התשובה האמיתית של הכלי (לפחות כמה שורות)"
    assert set(ai_flagged) == {"analysis.py", "test_analysis.py", "report.py"}, "ai_flagged: שלושה קבצים: analysis.py, test_analysis.py, report.py"
    assert all(isinstance(v, bool) for v in ai_flagged.values()), "ai_flagged: לכל קובץ כתבו True או False (האם הכלי סימן אותו כבעייתי)"
    return True

check_d5() and print("✓ שלב 5: התיעוד מוכן")

✓ שלב 5: התיעוד מוכן


<div dir="rtl">

נציג זה לצד זה: מה **הכללים שלכם** תפסו, ומה **Gemini** תפס:

</div>

In [52]:
my_flagged = {f: any(file == f for _, file, _ in review_diff(DIFF)) for f in ai_flagged}
pd.DataFrame({"my_rules": my_flagged, "gemini": ai_flagged})

,my_rules,gemini
analysis.py,False,False
test_analysis.py,True,True
report.py,True,True


<div dir="rtl">

**(א)** מה **Gemini** מצא שהכללים שלכם **לא** תפסו (או להפך)? האם הוא טעה או המציא משהו? כל אחד משלושת הכללים משויך לאחת משלוש השאלות מהשקף (קבצים / התנהגות / סודות): איזה כלל לאיזו שאלה?

**תשובה:** הכלי הוסיף שלא נבדקה בדיקה חדשה ל-top_customers, וציין לאמת את החבילה לפני התקנה: דברים שהכללים שלנו לא בודקים. הכללים תופסים בדיוק את מה שהגדרנו, תמיד, ובאותו אופן. התאמה לשאלות: deleted_test = התנהגות (משהו השתנה בשקט, הבדיקה שמגינה נעלמה), new_import = קבצים/תלויות (נוסף משהו שלא התבקש), hardcoded = סודות ומקרי קצה.

</div>

<div dir="rtl">

**(ב)** מדוע לא מסתפקים בתשובה של הכלי ("סקור לי את ה-diff"), וגם לא רק בכללים שכתבתם? מה העיקרון מההרצאה (*"מה שהסוכן אומר אינו הוכחה"*) אומר על סקירה שעושה AI?

**תשובה:** גם סקירה של AI היא פלט של מודל: יכולה להחמיץ או להמציא, ובכל הרצה עשויה להיות שונה. כללים מפורשים הם בדיקה דטרמיניסטית וניתנת לחזרה אבל צרה. משלבים: כללים שתופסים את הידוע, AI שמרחיב וחושב על מה שלא חשבנו, ואדם שמחליט. ה-diff נשאר הכלי, והאחריות היא שלנו.

</div>

<div dir="rtl">

---

# חלק ה' - מסקנה (3 דק')

שלושה משפטים. כל משפט בנוי כך: **מה מימשתם ← איזו חוליה או איזה עיקרון מההרצאה זה מראה ← מה עושים בעבודה עם AI.**

1. **סוכן:** כתבתי לולאה של תכנון, פעולה והתבוננות עם מפעיל כלים שהופך שגיאה למידע, ועם max_steps ואישור לפי רמה, ולכן אני יודע שסוכן הוא לא מודל אלא מערכת שצריך לשים לה גבולות לפני שנותנים לה אוטונומיה.
2. **Git:** עשיתי commit לפני, גיליתי עם status מה הסוכן שינה מעבר למבוקש ושחזרתי רק את מה שלא התבקש, ולכן לפני כל ריצת סוכן אני שומר נקודת חזרה ועובד על ענף נפרד.
3. **סקירת diff:** כתבתי כללים שתופסים בדיקה שנמחקה, חבילה חדשה וערך קשיח, והשוויתי אותם ל-Gemini, ולכן קוראים diff לפני שמקבלים שינוי של סוכן, ולא סומכים על "סיימתי".

</div>

<div dir="rtl">

---

# בונוס

**1. זיהוי לולאה.** הוסיפו `detect_loop(steps, k=3)`: מחזירה `True` אם `k` הפעולות האחרונות ב-`steps` **זהות** (אותו `tool` ואותם `args`). חברו אותה ל-`run_agent` כך שהסוכן התקוע יעצור **לפני** `max_steps`, עם `status="loop"`. מה היתרון על `max_steps` בלבד?

**2. כלל רביעי.** הוסיפו ל-`review_diff` את הכלל `deleted_file`: קובץ שנמחק כולו (ב-diff מופיעה שורה `deleted file mode`). הריצו על `git diff` של `damaged` מחלק ג' (`_g(damaged, "diff")`): האם `products.csv` מסומן?

**3. סוכן אמיתי.** חזרו על מעבדות א' ו-ב' מהשקפים עם סוכן אמיתי (בעורך או בשורת פקודה, בסביבה שבה הוא זמין), הפעם **על פי הכלל**: commit לפני, ענף לניסוי, `git status` ו-diff אחרי, ו-`restore` למה שלא התבקש. תעדו בטבלה: מה הסוכן שינה מעבר למבוקש? (זה לא רץ ב-Colab.)

</div>

In [53]:
# תא לבונוס 1

In [54]:
# תא לבונוס 2

In [55]:
# תא לבונוס 3

<div dir="rtl">

---

# בדיקה מסכמת - הריצו לפני ההגשה

התא הבא מריץ שוב את כל תאי הבדיקה. לפני ההגשה: **Runtime ← Restart session and run all**, וודאו שכל השורות כאן מסומנות `✓`.

</div>

In [56]:
for name, check in [("א' - execute_action", check_a1),
                    ("א' - run_agent", check_a2),
                    ("א' - תמלול הסוכן הזהיר", check_a2b),
                    ("א' - סוכן תקוע ו-max_steps", check_a3),
                    ("א' - needs_approval", check_a4),
                    ("א' - טבלת הרמות", check_a4b),
                    ("ב' - git", check_b1),
                    ("ב' - commit_all", check_b2),
                    ("ב' - start_branch", check_b3),
                    ("ב' - changed_files", check_b4),
                    ("ב' - unexpected", check_b5),
                    ("ב' - restore_file", check_b6),
                    ("ג' - diff_stat", check_c1),
                    ("ג' - count_commits, merge_branch", check_c2),
                    ("ד' - is_new_import", check_d2),
                    ("ד' - is_hardcoded", check_d3),
                    ("ד' - review_diff", check_d4),
                    ("ד' - תיעוד Gemini", check_d5)]:
    try:
        check()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"✗ {name}  ← עדיין לא מומש")
    except AssertionError as e:
        print(f"✗ {name}  ← {e}")
    except Exception as e:
        print(f"✗ {name}  ← שגיאה בקוד: {type(e).__name__}: {e}")

✓ א' - execute_action
✓ א' - run_agent
✓ א' - תמלול הסוכן הזהיר
✓ א' - סוכן תקוע ו-max_steps
✓ א' - needs_approval
✓ א' - טבלת הרמות


✓ ב' - git


✓ ב' - commit_all


✓ ב' - start_branch


✓ ב' - changed_files
✓ ב' - unexpected


✓ ב' - restore_file
✓ ג' - diff_stat


✓ ג' - count_commits, merge_branch
✓ ד' - is_new_import
✓ ד' - is_hardcoded
✓ ד' - review_diff
✓ ד' - תיעוד Gemini
